In [0]:
%sql
use catalog `e-commerce`;
use schema silver;

In [0]:
from pyspark.sql.functions import col,sum,to_date,lower,when,datediff,first,lower,upper,avg,coalesce

In [0]:
orders=spark.table("`e-commerce`.bronze.orders")
orders=orders.dropDuplicates(['order_id'])

orders.select([sum(col(c).isNull().cast("int")).alias(c) for c in orders.columns]).display()

In [0]:
orders.columns

In [0]:
orders=orders.withColumn("order_purchase_date",to_date(col("order_purchase_timestamp")))

In [0]:
#standardising date columns
date_cols=["order_purchase_date","order_approved_at","order_delivered_carrier_date","order_delivered_customer_date","order_estimated_delivery_date"]
for c in date_cols:
  orders=orders.withColumn(c,col(c).cast("timestamp"))

In [0]:
#standardising order status
orders=orders.withColumn("order_status",lower(col("order_status")))

In [0]:
#extracting columns
orders=orders.withColumn("is_approved",when(col("order_approved_at").isNotNull(),1).otherwise(0))

orders=orders.withColumn("is_delivered",when(col("order_delivered_customer_date").isNotNull(),1).otherwise(0))

orders=orders.withColumn("is_shipped",when(col("order_delivered_carrier_date").isNotNull(),1).otherwise(0))

In [0]:
#extract order date
orders = orders.withColumn(
    "order_date",
    to_date(col("order_purchase_timestamp"))
)

In [0]:
#late delivery
orders=orders.withColumn("is_late",when(col("order_delivered_customer_date")>col("order_estimated_delivery_date"),1).otherwise(0))

#delivery before purchase
orders=orders.withColumn("invalid_delivery",when(col("order_delivered_customer_date")<col("order_purchase_timestamp"),1).otherwise(0))

#number of days taken to deliver
orders=orders.withColumn("delivery_days",datediff(col("order_delivered_customer_date"),col("order_purchase_timestamp")))



In [0]:
#save table to silver db
orders.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("`e-commerce`.silver.orders")

In [0]:
order_reviews = spark.table("`e-commerce`.bronze.order_reviews")

In [0]:
display(order_reviews)

In [0]:
order_reviews=order_reviews.dropDuplicates(['review_id'])

order_reviews.select([sum(col(c).isNull().cast("int")).alias(c) for c in order_reviews.columns]).display()

In [0]:
#filter non null columns for ids and score
order_reviews = order_reviews.filter(col("review_id").isNotNull())
order_reviews = order_reviews.filter(col("order_id").isNotNull())
order_reviews = order_reviews.filter((col("review_score").isNotNull())&(col("review_score") >= 1) &
    (col("review_score") <= 5))

In [0]:
#data types
order_reviews = order_reviews.withColumn("review_score", col("review_score").cast("int"))
order_reviews = order_reviews.withColumn("review_creation_date", col("review_creation_date").cast("timestamp"))
order_reviews = order_reviews.withColumn("review_answer_timestamp", col("review_answer_timestamp").cast("timestamp"))

In [0]:
#fill missing texts in reviews
order_reviews = order_reviews.fillna({"review_comment_title": "no_title"})
order_reviews = order_reviews.fillna({"review_comment_message": "no_message"})

In [0]:
order_reviews.write.format("delta").mode("overwrite").saveAsTable("`e-commerce`.silver.order_reviews")

In [0]:
order_payments = spark.table("`e-commerce`.bronze.order_payments")
display(order_payments)

In [0]:
order_payments = order_payments.dropDuplicates()
order_payments.select([sum(col(c).isNull().cast("int")).alias(c) for c in order_payments.columns]).display()

In [0]:
#data types
order_payments = order_payments.withColumn("payment_value", col("payment_value").cast("double"))
order_payments = order_payments.withColumn("payment_installments", col("payment_installments").cast("int"))

In [0]:
#multiple payments
order_payments = order_payments.groupBy("order_id").agg(
    sum("payment_value").alias("total_payment"),
    first("payment_type").alias("payment_type")
)

In [0]:
order_payments.write.format("delta").mode("overwrite").saveAsTable("`e-commerce`.silver.order_payments")

In [0]:
order_items = spark.table("`e-commerce`.bronze.order_items")
display(order_items)


In [0]:
order_items = order_items.dropDuplicates(['order_id','order_item_id'])
order_items.select([sum(col(c).isNull().cast("int")).alias(c) for c in order_items.columns]).display()

In [0]:
#data types
order_items = order_items.withColumn("price", col("price").cast("double"))
order_items = order_items.withColumn("freight_value", col("freight_value").cast("double"))
order_items = order_items.withColumn("shipping_limit_date", col("shipping_limit_date").cast("timestamp"))

In [0]:
#total revenue
order_items=order_items.withColumn("total_price",col("price")+col("freight_value"))


In [0]:
order_items.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("`e-commerce`.silver.order_items")

In [0]:
customers = spark.table("`e-commerce`.bronze.customers")
display(customers)


In [0]:
customers=customers.dropDuplicates(['customer_id'])
customers.select([sum(col(c).isNull().cast("int")).alias(c) for c in customers.columns]).display()


In [0]:
#standardise text
customers = customers.withColumn("customer_city", lower(col("customer_city")))
customers = customers.withColumn("customer_state", upper(col("customer_state")))

In [0]:
customers.write.format("delta").mode("overwrite") .option("overwriteSchema", "true").saveAsTable("silver.customers")

In [0]:
geolocation=spark.table("`e-commerce`.bronze.geolocation")
display(geolocation)

geolocation=geolocation.dropDuplicates(['geolocation_zip_code_prefix'])
geolocation.select([sum(col(c).isNull().cast("int")).alias(c) for c in geolocation.columns]).display()

In [0]:
geolocation = geolocation.withColumn("geolocation_city", lower(col("geolocation_city")))
geolocation = geolocation.withColumn("geolocation_state", upper(col("geolocation_state")))

In [0]:
#group by zip code
geolocation= geolocation.groupBy("geolocation_zip_code_prefix").agg(
    avg("geolocation_lat").alias("avg_lat"),
    avg("geolocation_lng").alias("avg_lng"),
    first("geolocation_city").alias("city"),
    first("geolocation_state").alias("state")
)

In [0]:
#clean naming
geolocation = geolocation.withColumnRenamed("geolocation_zip_code_prefix", "zip_code")

In [0]:
geolocation.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("`e-commerce`.silver.geolocation")

In [0]:
products=spark.table("`e-commerce`.bronze.products")
display(products)

products=products.dropDuplicates(['product_id'])
products.select([sum(col(c).isNull().cast("int")).alias(c) for c in products.columns]).display()

In [0]:
products=products.fillna({"product_category_name": "unknown",
    "product_name_lenght": 0,
    "product_description_lenght": 0,
    "product_photos_qty": 0,
    "product_weight_g": 0,
    "product_length_cm": 0,
    "product_height_cm": 0,
    "product_width_cm": 0})

In [0]:
products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.products")

In [0]:
sellers=spark.table("`e-commerce`.bronze.sellers")
display(sellers)

sellers=sellers.dropDuplicates(['seller_id'])
sellers=sellers.filter(col("seller_id").isNotNull())

In [0]:
sellers=sellers.fillna({"seller_city":"unknown","seller_state":"unknown"})

In [0]:
sellers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.sellers")

In [0]:
product_category=spark.table("`e-commerce`.bronze.product_category")
display(product_category)



In [0]:
products=products.join(product_category,on='product_category_name',how='left')

In [0]:
#english category
products = products.withColumn(
    "product_category_final",
    coalesce(col("product_category_name_english"), col("product_category_name"))
)

In [0]:
products = products.drop("product_category_name_english")

In [0]:
products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.products")